[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/duoan/TorchCode/blob/master/templates/16_cross_entropy.ipynb)

# 🟢 Easy: Cross-Entropy Loss

Implement **cross-entropy loss** from scratch.

$$\text{CE}(x, y) = -\log\frac{e^{x_y}}{\sum_j e^{x_j}}$$

### Signature
```python
def cross_entropy_loss(logits: Tensor, targets: Tensor) -> Tensor:
    # logits: (B, C) float, targets: (B,) long indices
    # Returns: scalar loss (mean over batch)
```

### Rules
- Do NOT use `F.cross_entropy` or `nn.CrossEntropyLoss`
- Must be numerically stable (use logsumexp trick)

In [4]:
# Install torch-judge in Colab (no-op in JupyterLab/Docker)
try:
    import google.colab
    get_ipython().run_line_magic('pip', 'install -q torch-judge')
except ImportError:
    pass


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.5/48.5 kB 4.1 MB/s eta 0:00:00


In [5]:
import torch

In [28]:
# ✏️ YOUR IMPLEMENTATION HERE

def cross_entropy_loss(logits, targets):
    #logprobs = log(softmax(X)) = log(exp(X)) - log(sum(exp(X))) = logits - logsumexp(x)
    #to compute a numerically stable logsumexp() we should again reduce each exponent of the X's maximum to avoiud overflow (so after - max exponents would all be <= 0)
    #sum(exp(X)) = exp(max(X)) * sum(X - max(X)) --> logsumexp(X) = max(X) + logsumexp(X - max(X))

    #after we got logprobs, we should return -logprobs[targets]
    B = logits.shape[0]
    max_el = torch.max(logits, dim = -1, keepdim=True).values
    _logsumexp_shifted = torch.log(torch.sum(torch.exp(logits - max_el), dim = -1, keepdim=True))
    _logsumexp = max_el + _logsumexp_shifted
    logprobs = logits - _logsumexp
    target_logprobs = logprobs[torch.arange(B), targets] #row = batch_index, col = target_index
    return torch.mean(-target_logprobs)



In [29]:
# 🧪 Debug
logits = torch.randn(4, 10)
targets = torch.randint(0, 10, (4,))
print('Loss:', cross_entropy_loss(logits, targets))
print('Ref: ', torch.nn.functional.cross_entropy(logits, targets))

Loss: tensor(2.6051)
Ref:  tensor(2.6051)


In [30]:
# ✅ SUBMIT
from torch_judge import check
check('cross_entropy')


🧪 Testing: Cross-Entropy Loss (Easy)
──────────────────────────────────────────────────
  ✅ [1/4] Matches F.cross_entropy (9.3ms)
  ✅ [2/4] Numerical stability (0.8ms)
  ✅ [3/4] Scalar output (0.3ms)
  ✅ [4/4] Gradient flow (27.6ms)
──────────────────────────────────────────────────
  🎉 All 4 tests passed! (38.0ms total)
  Progress saved. Run status() to see your dashboard.

